# S3 — Single-Obs Sweep Diagnostics
**Cell 1** — all imports, helpers, and plotting functions. Do not modify.

**Cell 2** — data loading. Set `mode='single'` or `mode='all'` and configure paths.

**Plot cells** — one per figure. Only configuration and function calls.

In [ ]:
# =============================================================================
# CELL 1 — IMPORTS, HELPERS, AND ALL PLOTTING FUNCTIONS
# Do not modify this cell. Configuration lives in each plot cell.
# =============================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

# ---------------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'figure.dpi':     150,
    'font.size':      10,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'legend.fontsize':8,
    'xtick.labelsize':8,
    'ytick.labelsize':8,
})

# ---------------------------------------------------------------------------
# Colour palette — consistent across all figures
# Extend as needed for more ntemp values
# ---------------------------------------------------------------------------
METHOD_COLORS = {
    'prior':    'black',
    'TEnKF_1':  '#1f77b4',
    'TEnKF_2':  '#2ca02c',
    'TEnKF_3':  '#d62728',
    'TEnKF_4':  '#17becf',
    'TEnKF_5':  '#8c564b',
    'TEnKF_6':  '#e377c2',
    'TEnKF_7':  '#7f7f7f',
    'TEnKF_8':  '#bcbd22',
    'TEnKF_9':  '#9467bd',
    'TEnKF_10': '#ff7f0e',
    'AOEI':     'DarkGrey',
}
METHOD_LABELS = {
    'prior':    'Prior (O−B)',
    **{f'TEnKF_{n}': f'TEnKF Nt={n}' for n in range(1, 11)},
    'AOEI':     'AOEI',
}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _stats_label(name, arr):
    """Legend label with mean, std, n."""
    return f"{name}\nμ={np.nanmean(arr):.2f}  σ={np.nanstd(arr):.2f}  n={len(arr)}"


def _get_method_df(df, method_key):
    """Return sub-DataFrame for a method key like 'TEnKF_3' or 'AOEI' or 'prior'."""
    if method_key == 'prior':
        # prior is the same for all combos at a point — use ntemp==1 as canonical
        return df[(df['method'] == 'TEnKF') & (df['ntemp'] == 1)]
    if method_key.startswith('TEnKF'):
        nt = int(method_key.split('_')[1])
        return df[(df['method'] == 'TEnKF') & (df['ntemp'] == nt)]
    return df[df['method'] == method_key]


def _save_or_show(fig, savepath):
    if savepath:
        os.makedirs(os.path.dirname(savepath), exist_ok=True)
        fig.savefig(savepath, bbox_inches='tight')
        print(f'Saved: {savepath}')
    plt.show()


def _scatter_to_grid_xy(x_km, y_km, values, collapse='mean', bins=80):
    """Bin scattered (x,y,value) onto a 2D XY grid. Returns x_edges, y_edges, grid."""
    x_edges = np.linspace(x_km.min(), x_km.max(), bins + 1)
    y_edges = np.linspace(y_km.min(), y_km.max(), bins + 1)
    grid    = np.full((bins, bins), np.nan)
    cnt     = np.zeros((bins, bins))
    acc     = np.full((bins, bins), -np.inf if collapse == 'max' else 0.0)

    xi = np.clip(np.digitize(x_km, x_edges) - 1, 0, bins - 1)
    yi = np.clip(np.digitize(y_km, y_edges) - 1, 0, bins - 1)

    for ix, iy, v in zip(xi, yi, values):
        if np.isnan(v):
            continue
        if collapse == 'max':
            acc[ix, iy] = max(acc[ix, iy], v)
        else:
            acc[ix, iy] += v
            cnt[ix, iy] += 1

    if collapse == 'mean':
        mask = cnt > 0
        acc[mask]  = acc[mask] / cnt[mask]
        acc[~mask] = np.nan
    else:
        acc[acc == -np.inf] = np.nan

    return x_edges, y_edges, acc.T


# ---------------------------------------------------------------------------
# PLOT 1 — Departure histogram
# ---------------------------------------------------------------------------

def plot_departure_histogram(df, methods,
                              hist_bins=50, hist_range=(-20, 20),
                              title='Distribution of Observation Departures (O−B, O−A)',
                              savepath=None,density=True):
    fig, ax = plt.subplots(figsize=(12, 5))
    for mk in methods:
        sub = _get_method_df(df, mk)
        arr = sub['dep_b' if mk == 'prior' else 'dep_a'].values
        #filter out departur with values of zero
        arr = arr[~np.isclose(arr, 0)]
        ax.hist(arr, bins=hist_bins, range=hist_range,
                density=density, histtype='step', linewidth=2,
                color=METHOD_COLORS[mk],
                linestyle='--' if mk == 'prior' else '-',
                label=_stats_label(METHOD_LABELS[mk], arr))
    ax.axvline(0, color='gray', linewidth=1)
    ax.set(title=title, xlabel='Departure [dBZ]', ylabel='Probability Density',
           xlim=hist_range)
    ax.legend(loc='upper left')
    ax.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    _save_or_show(fig, savepath)


# ---------------------------------------------------------------------------
# PLOT 2 — Hexbin: x_col vs RMSE reduction (rmse_a_obs_w - rmse_f_obs_w)
# Negative y = improvement over prior.
# ---------------------------------------------------------------------------

def plot_hexbin_rmse_reduction(df, methods, x_col,
                                x_label,
                                x_range=(-20, 20),
                                y_range=(-15, 15),
                                gridsize=40,
                                one_figure=True,
                                title='',
                                savepath=None):
    """
    y-axis: rmse_a_obs_w - rmse_f_obs_w  (negative = improvement)
    x_col:  any column in df, typically 'dep_b' or 'truth_hx_mean_local'
    """
    # exclude 'prior' from methods — it has no analysis
    methods = [m for m in methods if m != 'prior']
    n = len(methods)
    norm = mcolors.LogNorm()

    if one_figure:
        ncols = min(n, 3)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5 * ncols, 4 * nrows),
                                  squeeze=False)
        fig.suptitle(title, fontsize=12)
        axf = iter(axes.flat)
        figs = [fig]
    else:
        figs = []

    for mk in methods:
        sub  = _get_method_df(df, mk)
        x    = sub[x_col].values
        drmse = (sub['rmse_a_obs_w'] - sub['rmse_f_obs_w']).values
        mask = ~(np.isnan(x) | np.isnan(drmse))
        x, drmse = x[mask], drmse[mask]

        if one_figure:
            ax = next(axf)
        else:
            fig, ax = plt.subplots(figsize=(6, 5))
            fig.suptitle(f"{title} — {METHOD_LABELS[mk]}", fontsize=11)
            figs.append(fig)

        hb = ax.hexbin(x, drmse, gridsize=gridsize,
                        extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
                        cmap='plasma', norm=norm, mincnt=1)
        ax.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.7)
        plt.colorbar(hb, ax=ax, label='count')
        ax.set_title(METHOD_LABELS[mk])
        ax.set_xlabel(x_label)
        ax.set_ylabel('RMSE change (analysis − prior) [dBZ]')
        ax.grid(True, linestyle=':', alpha=0.4)

    if one_figure:
        for ax in axf:
            ax.set_visible(False)

    for fig in figs:
        plt.tight_layout()
        _save_or_show(fig, savepath)


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# PLOT 4/5 — Binned lines: x_col bins → mean y per method
# mode: 'single' shows one y_col; 'triple' shows weighted/unweighted/point
# ---------------------------------------------------------------------------

def plot_binned_lines(df, methods, x_col,
                      y_col=None,
                      mode='triple',
                      x_label='', y_label='',
                      n_bins=15, x_range=None, show_std=True,
                      hline=None, title='', savepath=None):
    """
    mode='single': plot one y_col per method (use y_col=).
    mode='triple': three sub-plots side by side:
                   weighted (rmse_*_obs_w), unweighted (rmse_*_obs_u),
                   point (err_*_obs_pt).
    y_col is ignored in triple mode.
    RMSE change = analysis - forecast for each variant.
    """
    if mode == 'triple':
        variants = [
            ('Weighted zone RMSE change',   'rmse_f_obs_w',  'rmse_a_obs_w'),
            ('Unweighted zone RMSE change', 'rmse_f_obs_u',  'rmse_a_obs_u'),
            ('Point error change',          'err_f_obs_pt',  'err_a_obs_pt'),
        ]
        fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
        fig.suptitle(title, fontsize=12)
        for ax, (vtitle, fcol, acol) in zip(axes, variants):
            _binned_axis(ax, df, methods, x_col, fcol, acol,
                         x_range, n_bins, show_std, hline, x_label)
            ax.set_title(vtitle)
        plt.tight_layout()
        _save_or_show(fig, savepath)
    else:
        fig, ax = plt.subplots(figsize=(11, 5))
        all_x = pd.concat([_get_method_df(df, m)[x_col] for m in methods
                            if m != 'prior'], ignore_index=True).dropna()
        lo = x_range[0] if x_range else float(all_x.quantile(0.01))
        hi = x_range[1] if x_range else float(all_x.quantile(0.99))
        edges   = np.linspace(lo, hi, n_bins + 1)
        centers = 0.5 * (edges[:-1] + edges[1:])
        for mk in methods:
            sub = _get_method_df(df, mk)
            x   = sub[x_col].values
            y   = sub[y_col].values
            means = np.full(n_bins, np.nan)
            stds  = np.full(n_bins, np.nan)
            for b in range(n_bins):
                msk = (x >= edges[b]) & (x < edges[b+1]) & ~np.isnan(y)
                if msk.sum() >= 5:
                    means[b] = np.nanmean(y[msk])
                    stds[b]  = np.nanstd(y[msk])
            valid = ~np.isnan(means)
            ax.plot(centers[valid], means[valid], color=METHOD_COLORS[mk],
                    linestyle='--' if mk=='prior' else '-', linewidth=2,
                    label=METHOD_LABELS[mk], marker='o', markersize=4)
            if show_std:
                ax.fill_between(centers[valid],
                                means[valid]-stds[valid], means[valid]+stds[valid],
                                color=METHOD_COLORS[mk], alpha=0.12)
        if hline is not None:
            ax.axhline(hline, color='gray', linewidth=1, linestyle=':')
        ax.set(title=title, xlabel=x_label, ylabel=y_label)
        ax.legend(loc='best', ncol=2)
        ax.grid(True, linestyle=':', alpha=0.5)
        plt.tight_layout()
        _save_or_show(fig, savepath)


def _binned_axis(ax, df, methods, x_col, f_col, a_col,
                 x_range, n_bins, show_std, hline, x_label):
    """Helper: draw one binned-line axis for a given f/a column pair."""
    all_x = pd.concat([_get_method_df(df, m)[x_col] for m in methods
                        if m != 'prior'], ignore_index=True).dropna()
    lo = x_range[0] if x_range else float(all_x.quantile(0.01))
    hi = x_range[1] if x_range else float(all_x.quantile(0.99))
    edges   = np.linspace(lo, hi, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    for mk in methods:
        sub = _get_method_df(df, mk)
        x   = sub[x_col].values
        y   = (sub[a_col] - sub[f_col]).values
        means = np.full(n_bins, np.nan)
        stds  = np.full(n_bins, np.nan)
        for b in range(n_bins):
            msk = (x >= edges[b]) & (x < edges[b+1]) & ~np.isnan(y)
            if msk.sum() >= 5:
                means[b] = np.nanmean(y[msk])
                stds[b]  = np.nanstd(y[msk])
        valid = ~np.isnan(means)
        ax.plot(centers[valid], means[valid], color=METHOD_COLORS[mk],
                linestyle='--' if mk=='prior' else '-', linewidth=2,
                label=METHOD_LABELS[mk], marker='o', markersize=3)
        if show_std:
            ax.fill_between(centers[valid],
                            means[valid]-stds[valid], means[valid]+stds[valid],
                            color=METHOD_COLORS[mk], alpha=0.10)
    if hline is not None:
        ax.axhline(hline, color='gray', linewidth=1, linestyle=':')
    ax.set_xlabel(x_label)
    ax.set_ylabel('RMSE change (analysis - prior) [dBZ]')
    ax.legend(loc='best', ncol=2, fontsize=7)
    ax.grid(True, linestyle=':', alpha=0.5)


def plot_spatial_xy(df, methods, value_col,
                    collapse='mean',
                    bins=80,
                    cmap='plasma', vmin=None, vmax=None,
                    cbar_label='',
                    one_figure=True,
                    title='',
                    savepath=None):
    """
    XY spatial map, one panel per method.
    value_col: str or dict {method_key: col_name}
    one_figure: all methods side by side in one figure, else one figure per method.
    """
    n = len(methods)

    if one_figure:
        fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5), squeeze=False)
        fig.suptitle(title, fontsize=12)
        figs = [(fig, axes[0])]
    else:
        figs = []

    for idx, mk in enumerate(methods):
        sub  = _get_method_df(df, mk)
        col  = value_col[mk] if isinstance(value_col, dict) else value_col
        vals = sub[col].values
        xk   = sub['x_km'].values
        yk   = sub['y_km'].values

        x_edges, y_edges, grid = _scatter_to_grid_xy(xk, yk, vals,
                                                      collapse=collapse, bins=bins)

        if one_figure:
            ax = figs[0][1][idx]
        else:
            fig, ax = plt.subplots(figsize=(6, 5))
            fig.suptitle(f"{title} — {METHOD_LABELS[mk]}", fontsize=11)
            figs.append((fig, np.array([[ax]])[0]))

        im = ax.pcolormesh(x_edges, y_edges, grid,
                            cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
        plt.colorbar(im, ax=ax, label=cbar_label, shrink=0.85, pad=0.02)
        ax.set_title(METHOD_LABELS[mk])
        ax.set_xlabel('x [km]')
        ax.set_ylabel('y [km]')

    for fig, _ in figs:
        plt.tight_layout()
        _save_or_show(fig, savepath)


# ---------------------------------------------------------------------------
# PLOT 10 — 2D hexbin: dep_b vs storm intensity, colored by mean RMSE change
# ---------------------------------------------------------------------------

def plot_2d_hexbin_rmse(df, methods,
                         x_range=(-20, 20),
                         y_range=(0, 60),
                         gridsize=30,
                         vmin=-15, vmax=15,
                         one_figure=True,
                         title='Prior departure × storm intensity → mean RMSE change',
                         savepath=None):
    """
    2D hexbin: x = dep_b, y = truth_hx_mean_local
    Color = mean(rmse_a_obs_w - rmse_f_obs_w) per hex bin.
    Negative color = improvement over prior.
    One panel per method.
    """
    methods = [m for m in methods if m != 'prior']
    n = len(methods)

    if one_figure:
        ncols = min(n, 3)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5.5 * ncols, 4.5 * nrows),
                                  squeeze=False)
        fig.suptitle(title, fontsize=12)
        axf = iter(axes.flat)
        figs = [fig]
    else:
        figs = []

    for mk in methods:
        sub   = _get_method_df(df, mk)
        x     = sub['dep_b'].values
        y     = sub['hx_dbz_local_mean_w'].values
        drmse = (sub['rmse_a_obs_w'] - sub['rmse_f_obs_w']).values
        mask  = ~(np.isnan(x) | np.isnan(y) | np.isnan(drmse))
        x, y, drmse = x[mask], y[mask], drmse[mask]

        if one_figure:
            ax = next(axf)
        else:
            fig, ax = plt.subplots(figsize=(6.5, 5))
            fig.suptitle(f"{title} — {METHOD_LABELS[mk]}", fontsize=11)
            figs.append(fig)

        # use hexbin with reduce_C_function=np.mean to color by mean drmse
        hb = ax.hexbin(x, y, C=drmse, gridsize=gridsize,
                        extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
                        reduce_C_function=np.mean,
                        cmap='RdBu_r', vmin=vmin, vmax=vmax, mincnt=3)
        ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
        plt.colorbar(hb, ax=ax, label='Mean RMSE change [dBZ]')
        ax.set_title(METHOD_LABELS[mk])
        ax.set_xlabel('Prior departure (O−B) [dBZ]')
        ax.set_ylabel('Storm intensity in zone [dBZ]')
        ax.grid(True, linestyle=':', alpha=0.3)

    if one_figure:
        for ax in axf:
            ax.set_visible(False)

    for fig in figs:
        plt.tight_layout()
        _save_or_show(fig, savepath)


# ---------------------------------------------------------------------------
# PLOT 11 — same 2D hexbin but colored by count
# ---------------------------------------------------------------------------

def plot_2d_hexbin_count(df, methods,
                          x_range=(-20, 20),
                          y_range=(0, 60),
                          gridsize=30,
                          one_figure=True,
                          title='Prior departure × storm intensity → point count',
                          savepath=None):
    """
    Same axes as plot_2d_hexbin_rmse but color = log count.
    Reveals where most of the data lives.
    Uses 'prior' or any one method — data is identical across methods.
    Pass a single-element list to avoid redundant panels.
    """
    methods = [m for m in methods if m != 'prior']
    n = len(methods)

    if one_figure:
        ncols = min(n, 3)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5.5 * ncols, 4.5 * nrows),
                                  squeeze=False)
        fig.suptitle(title, fontsize=12)
        axf = iter(axes.flat)
        figs = [fig]
    else:
        figs = []

    for mk in methods:
        sub  = _get_method_df(df, mk)
        x    = sub['dep_b'].values
        y    = sub['hx_dbz_local_mean_w'].values
        mask = ~(np.isnan(x) | np.isnan(y))
        x, y = x[mask], y[mask]

        if one_figure:
            ax = next(axf)
        else:
            fig, ax = plt.subplots(figsize=(6.5, 5))
            fig.suptitle(f"{title} — {METHOD_LABELS[mk]}", fontsize=11)
            figs.append(fig)

        hb = ax.hexbin(x, y, gridsize=gridsize,
                        extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
                        cmap='plasma', norm=mcolors.LogNorm(), mincnt=1)
        ax.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.6)
        plt.colorbar(hb, ax=ax, label='count')
        ax.set_title(METHOD_LABELS[mk])
        ax.set_xlabel('Prior departure (O−B) [dBZ]')
        ax.set_ylabel('Storm intensity in zone [dBZ]')
        ax.grid(True, linestyle=':', alpha=0.3)

    if one_figure:
        for ax in axf:
            ax.set_visible(False)

    for fig in figs:
        plt.tight_layout()
        _save_or_show(fig, savepath)


# ---------------------------------------------------------------------------
# PLOT 12 — consecutive tempering improvement
# Each panel: rmse_a(Nt=n) - rmse_a(Nt=n+1)  (positive = n+1 is better)
# ---------------------------------------------------------------------------

def _build_consecutive_diffs(df, ntemp_sequence):
    """
    For each consecutive pair in ntemp_sequence, compute per-point
    rmse_a_obs_w(Nt=n) - rmse_a_obs_w(Nt=n+1).

    Alignment assumption: rows for each ntemp group are in the same
    point order (guaranteed since _process_point iterates points
    identically for all combos). Multiple truth members are handled
    by sorting on (i, j, k) within each ntemp group before aligning.

    Returns list of dicts: {label, dep_b, truth_hx_mean_local, drmse}
    """
    df_t = df[df['method'] == 'TEnKF'].copy()
    pairs = list(zip(ntemp_sequence[:-1], ntemp_sequence[1:]))
    results = []

    for n1, n2 in pairs:
        d1 = df_t[df_t['ntemp'] == n1].reset_index(drop=True)
        d2 = df_t[df_t['ntemp'] == n2].reset_index(drop=True)
        if len(d1) != len(d2):
            print(f'  WARNING: Nt={n1} ({len(d1)} rows) != Nt={n2} ({len(d2)} rows) — skipping')
            continue
        results.append(dict(
            label=f'Nt={n1}→{n2}',
            dep_b=d1['dep_b'].values,
            storm=d1['hx_dbz_local_mean_w'].values,
            drmse=(d1['rmse_a_obs_w'] - d2['rmse_a_obs_w']).values,
        ))
    return results


def plot_2d_hexbin_tempering_steps(df, ntemp_sequence,
                                    x_range=(-20, 20),
                                    y_range=(0, 60),
                                    gridsize=30,
                                    vmin=-5, vmax=5,
                                    one_figure=True,
                                    title='Marginal improvement per tempering step '
                                          '(positive = adding step helps)',
                                    savepath=None):
    """
    One panel per consecutive Nt pair.
    Color = mean( rmse_a(Nt=n) - rmse_a(Nt=n+1) ) per hex bin.
    Positive (red) = adding one more tempering step improves the analysis.
    Negative (blue) = adding one more step makes things worse.
    """
    diffs = _build_consecutive_diffs(df, ntemp_sequence)
    n = len(diffs)
    if n == 0:
        print('No pairs to plot.')
        return

    if one_figure:
        ncols = min(n, 3)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5.5 * ncols, 4.5 * nrows),
                                  squeeze=False)
        fig.suptitle(title, fontsize=11)
        axf = iter(axes.flat)
        figs = [fig]
    else:
        figs = []

    for d in diffs:
        mask = ~(np.isnan(d['dep_b']) | np.isnan(d['storm']) | np.isnan(d['drmse']))
        x, y, c = d['dep_b'][mask], d['storm'][mask], d['drmse'][mask]

        if one_figure:
            ax = next(axf)
        else:
            fig, ax = plt.subplots(figsize=(6.5, 5))
            fig.suptitle(f"{title} — {d['label']}", fontsize=11)
            figs.append(fig)

        hb = ax.hexbin(x, y, C=c, gridsize=gridsize,
                        extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
                        reduce_C_function=np.mean,
                        cmap='RdBu_r', vmin=vmin, vmax=vmax, mincnt=3)
        ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
        plt.colorbar(hb, ax=ax, label='RMSE change [dBZ]')
        ax.set_title(d['label'])
        ax.set_xlabel('Prior departure (O−B) [dBZ]')
        ax.set_ylabel('Storm intensity in zone [dBZ]')
        ax.grid(True, linestyle=':', alpha=0.3)

    if one_figure:
        for ax in axf:
            ax.set_visible(False)

    for fig in figs:
        plt.tight_layout()
        _save_or_show(fig, savepath)




# ---------------------------------------------------------------------------
# PLOT 13 — all Nt values vs Nt=1
# Each panel: rmse_a(Nt=1) - rmse_a(Nt=n)  (positive/red = Nt=n beats Nt=1)
# ---------------------------------------------------------------------------

def plot_2d_hexbin_vs_nt1(df, ntemp_values,
                            x_range=(-40, 40),
                            y_range=(0, 60),
                            gridsize=30,
                            vmin=-10, vmax=10,
                            one_figure=True,
                            title='RMSE improvement vs TEnKF Nt=1 (red = Nt=n better than Nt=1)',
                            savepath=None):
    """
    One panel per ntemp value (excluding 1).
    Color = mean( rmse_a(Nt=1) - rmse_a(Nt=n) ) per hex bin.
    Red (positive) = Nt=n improves over Nt=1.
    Blue (negative) = Nt=n is worse than Nt=1.
    """
    df_t  = df[df['method'] == 'TEnKF'].copy()
    d_ref = df_t[df_t['ntemp'] == 1].reset_index(drop=True)
    ntvals = [n for n in ntemp_values if n != 1]
    n = len(ntvals)
    if n == 0:
        print('No ntemp values to compare against Nt=1.')
        return

    if one_figure:
        ncols = min(n, 3)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5.5 * ncols, 4.5 * nrows),
                                  squeeze=False)
        fig.suptitle(title, fontsize=11)
        axf = iter(axes.flat)
        figs = [fig]
    else:
        figs = []

    for nt in ntvals:
        d_n = df_t[df_t['ntemp'] == nt].reset_index(drop=True)
        if len(d_ref) != len(d_n):
            print(f'  WARNING: Nt=1 ({len(d_ref)} rows) != Nt={nt} ({len(d_n)} rows) -- skipping')
            continue

        x     = d_ref['dep_b'].values
        y     = d_ref['hx_dbz_local_mean_w'].values
        drmse = (d_ref['rmse_a_obs_w'] - d_n['rmse_a_obs_w']).values
        mask  = ~(np.isnan(x) | np.isnan(y) | np.isnan(drmse))
        x, y, drmse = x[mask], y[mask], drmse[mask]

        if one_figure:
            ax = next(axf)
        else:
            fig, ax = plt.subplots(figsize=(6.5, 5))
            fig.suptitle(f'Nt=1 vs Nt={nt}', fontsize=11)
            figs.append(fig)

        hb = ax.hexbin(x, y, C=drmse, gridsize=gridsize,
                        extent=[x_range[0], x_range[1], y_range[0], y_range[1]],
                        reduce_C_function=np.mean,
                        cmap='RdBu_r', vmin=vmin, vmax=vmax, mincnt=3)
        ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
        plt.colorbar(hb, ax=ax, label='RMSE(Nt=1) - RMSE(Nt=n) [dBZ]')
        ax.set_title(f'Nt=1 vs Nt={nt}')
        ax.set_xlabel('Prior departure (O-B) [dBZ]')
        ax.set_ylabel('Storm intensity in zone [dBZ]')
        ax.grid(True, linestyle=':', alpha=0.3)

    if one_figure:
        for ax in axf:
            ax.set_visible(False)

    for fig in figs:
        plt.tight_layout()
        _save_or_show(fig, savepath)



# ---------------------------------------------------------------------------
# SUMMARY TABLE
# ---------------------------------------------------------------------------

def plot_summary_table(df, methods,
                        title='Summary statistics per method (mean ± std)',
                        savepath=None):
    """
    Summary table with one metric column per diagnostic.
    First row  (prior)  : forecast quantities — dep_b, rmse_f_*, err_f_*, spread_f_obs.
    Other rows (methods): analysis quantities — dep_a, rmse_a_*, err_a_*, spread_a_obs.
    All rows compared against the prior baseline in row 1.
    """
    col_defs = [
        # (header,          prior_col,       analysis_col)
        ('Departure [dBZ]', 'dep_b',         'dep_a'),
        ('RMSE-w [dBZ]',    'rmse_f_obs_w',  'rmse_a_obs_w'),
        ('RMSE-u [dBZ]',    'rmse_f_obs_u',  'rmse_a_obs_u'),
        ('err-pt [dBZ]',    'err_f_obs_pt',  'err_a_obs_pt'),
        ('Spread [dBZ]',    'spread_f_obs',  'spread_a_obs'),
    ]
 
    rows = []
    for mk in methods:
        sub      = _get_method_df(df, mk)
        is_prior = (mk == 'prior')
        row      = {'Method': METHOD_LABELS[mk]}
        for header, f_col, a_col in col_defs:
            col = f_col if is_prior else a_col
            if col in sub.columns:
                v = sub[col].dropna()
                row[header] = f'{v.mean():.2f} ± {v.std():.2f}'
            else:
                row[header] = 'N/A'
        rows.append(row)
 
    tbl = pd.DataFrame(rows).set_index('Method')
 
    fig_h = max(2.5, 0.45 * (len(methods) + 2))
    fig_w = max(7,   1.7  *  len(col_defs))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
 
    t = ax.table(
        cellText  = tbl.values,
        rowLabels = tbl.index.tolist(),
        colLabels = tbl.columns.tolist(),
        cellLoc   = 'center',
        loc       = 'center',
    )
    t.auto_set_font_size(False)
    t.set_fontsize(9)
    t.auto_set_column_width(col=list(range(len(tbl.columns) + 1)))
 
    # header row
    for j in range(len(tbl.columns)):
        t[(0, j)].set_facecolor('#2c3e50')
        t[(0, j)].set_text_props(color='white', fontweight='bold')
 
    # row label colours by method
    for i, mk in enumerate(methods):
        t[(i + 1, -1)].set_facecolor(METHOD_COLORS[mk])
        t[(i + 1, -1)].set_text_props(color='white', fontweight='bold')
 
    ax.set_title(title, fontsize=11, pad=12)
    plt.tight_layout()
    _save_or_show(fig, savepath)
    return tbl
 



print('All helpers and plotting functions loaded.')

In [ ]:
# =============================================================================
# CELL 2 — DATA LOADING
# mode = 'single'    → one file for TIME
# mode = 'all'       → all truth members for TIME
# mode = 'all_times' → all truth members for every time in TIMES_LOAD
# =============================================================================

DATA_ROOT = '/home/jorge.gacitua/salidas/WRF_Single_Cycle_Assimilation/data'
TIMES_ALL = ['1800','1805','1810','1815','1820','1825','1830','1835','1840','1845','1850','1855']

# --- Selectors ---
TIME       = '1835'        # used for mode='single' and mode='all'
TIMES_LOAD = TIMES_ALL     # used for mode='all_times'; set to ['1835'] for one time
mode       = 'all_times'   # 'single' | 'all' | 'all_times'
FILE       = 'WS_sweep_test_sweep_Ne040_tm01.npz'  # only for mode='single'

def _sweep_dir(t):
    return f'{DATA_ROOT}/WS_sweep_test_{t}'

def load_sweep_npz(path):
    """Load one sweep npz into a DataFrame. Skips non-1D arrays and var_names."""
    d    = np.load(path, allow_pickle=True)
    cols = [k for k in d.files if d[k].ndim == 1 and k != 'var_names']
    return pd.DataFrame({k: d[k] for k in cols}), list(d['var_names'])

dfs, var_names = [], None

if mode == 'single':
    d, var_names = load_sweep_npz(f'{_sweep_dir(TIME)}/{FILE}')
    d['time'] = TIME
    dfs.append(d)
    print(f'Loaded 1 file for time {TIME}')

elif mode == 'all':
    for f in sorted(glob.glob(f'{_sweep_dir(TIME)}/WS_sweep_test_sweep_Ne040_tm*.npz')):
        if 'sweep' in os.path.basename(f):
            try:
                d, vn = load_sweep_npz(f)
                d['time'] = TIME
                dfs.append(d)
                if var_names is None:
                    var_names = vn
                print(f'  loaded {os.path.basename(f)}  ({len(d)} rows)')
            except Exception as e:
                print(f'  SKIP {os.path.basename(f)}: {e}')

elif mode == 'all_times':
    for t in TIMES_LOAD:
        n_t = 0
        for f in sorted(glob.glob(f'{_sweep_dir(t)}/WS_sweep_test_sweep_Ne040_tm*.npz')):
            if 'sweep' in os.path.basename(f):
                try:
                    d, vn = load_sweep_npz(f)
                    d['time'] = t
                    dfs.append(d)
                    n_t += 1
                    if var_names is None:
                        var_names = vn
                except Exception as e:
                    print(f'  SKIP {os.path.basename(f)}: {e}')
        print(f'  time {t}: {n_t} files')

df = pd.concat(dfs, ignore_index=True)
print(f'\nLoaded {len(df):,} rows | times: {sorted(df["time"].unique().tolist())}')
print(f'Methods:      {df["method"].unique()}')
print(f'ntemp values: {sorted(df["ntemp"].unique())}')
print(f'Variables:    {var_names}')

# Derived point-level error columns
df['err_f_obs_pt'] = (df['hxf_mean_obs'] - df['yo_clean']).abs()
df['err_a_obs_pt'] = (df['hxa_mean_obs'] - df['yo_clean']).abs()
df['derr_obs_pt']  = df['err_a_obs_pt'] - df['err_f_obs_pt']
print('Point-level error columns added.')


In [ ]:
print(df.columns.tolist())

In [ ]:
# --- align column names to plotting convention ---
df = df.rename(columns={
    'rmse_f_w_obs':     'rmse_f_obs_w',
    'rmse_a_w_obs':     'rmse_a_obs_w',
    'rmse_f_u_obs':     'rmse_f_obs_u',
    'rmse_a_u_obs':     'rmse_a_obs_u',
    'rmse_f_point_obs': 'rmse_f_obs_pt',
    'rmse_a_point_obs': 'rmse_a_obs_pt',
})

# truth_hx_mean_local: zone-mean truth reflectivity used as storm-intensity proxy.
# yo_clean is H(truth) at the obs point — best available approximation.
df['truth_hx_mean_local'] = df['yo']#df['yo_clean']

print('Column rename done. Available obs metrics:')
print([c for c in df.columns if 'obs' in c])

In [ ]:
# =============================================================================
# PLOT 1 — Departure histogram
# =============================================================================

methods    = ['prior', 'TEnKF_1', 'TEnKF_2', 'TEnKF_3','TEnKF_5','TEnKF_10', 'AOEI']#, 'TEnKF_3', 'TEnKF_5', , 'AOEI']
hist_bins  = 59
hist_range = (-60, 60)
savepath   = None

plot_departure_histogram(
    df, methods,
    hist_bins=hist_bins, hist_range=hist_range,
    title='Distribution of Observation Departures (O−B, O−A)',
    savepath=savepath,density=False
)

In [ ]:
# =============================================================================
# PLOT 2 — Hexbin: prior departure vs RMSE change (analysis - prior)
# Negative y = improvement. White dashed line at y=0.
# =============================================================================

methods    = ['TEnKF_1', 'TEnKF_2', 'TEnKF_3', 'TEnKF_5','TEnKF_10', 'AOEI']#, 'TEnKF_3', 'TEnKF_5', 'TEnKF_10', 'AOEI']
x_col      = 'dep_b'        # prior departure on x-axis
x_range    = (-60, 60)
y_range    = (-60, 60)       # RMSE change
gridsize   = 40
one_figure = True
savepath   = None

plot_hexbin_rmse_reduction(
    df, methods,
    x_col=x_col,
    x_label='Prior departure (O−B) [dBZ]',
    x_range=x_range, y_range=y_range,
    gridsize=gridsize, one_figure=one_figure,
    title='Prior departure vs RMSE change (analysis − prior)',
    savepath=savepath
)

In [ ]:
# =============================================================================
# PLOT 3 — Hexbin: storm intensity vs RMSE change
# =============================================================================

methods    = ['TEnKF_1', 'TEnKF_2', 'TEnKF_3','TEnKF_5','TEnKF_10', 'AOEI']#, 'TEnKF_3', 'TEnKF_5', 'TEnKF_10', 'AOEI']
x_col      = 'hx_dbz_local_mean_w'
x_range    = (-1, 20)
y_range    = (-40, 40)
gridsize   = 40
one_figure = True
savepath   = None

plot_hexbin_rmse_reduction(
    df, methods,
    x_col=x_col,
    x_label='Storm intensity in zone — rho-weighted mean H(truth) [dBZ]',
    x_range=x_range, y_range=y_range,
    gridsize=gridsize, one_figure=one_figure,
    title='Storm intensity vs RMSE change (analysis − prior)',
    savepath=savepath
)

In [ ]:
# =============================================================================
# PLOT 4 — Binned lines: prior departure bins → mean RMSE change ± std
# Shows how each method responds to different prior departure magnitudes.
# Negative = improvement over prior.
# =============================================================================

methods    = [ 'TEnKF_1', 'TEnKF_2', 'TEnKF_3','TEnKF_5','TEnKF_10', 'AOEI']#, 'TEnKF_3', 'TEnKF_5', 'TEnKF_10', 'AOEI']
x_col      = 'dep_b'
# y_col = None triggers automatic RMSE reduction (rmse_a_obs_w - rmse_f_obs_w)
y_col      = None
n_bins     = 20
x_range    = (-60, 60)
show_std   = True
savepath   = None

plot_binned_lines(
    df, methods,
    x_col=x_col, y_col=y_col,
    x_label='Prior departure (O−B) [dBZ]',
    y_label='Mean RMSE change (analysis − prior) [dBZ]',
    n_bins=n_bins, x_range=x_range,
    show_std=show_std, hline=0,
    title='Prior departure bins → mean RMSE change per method',
    savepath=savepath
)

In [ ]:
# =============================================================================
# PLOT 5 — Binned lines: storm intensity bins → mean RMSE change ± std
# Shows if methods perform differently inside vs outside the storm.
# =============================================================================

methods    = ['TEnKF_1', 'TEnKF_2', 'TEnKF_3','TEnKF_5','TEnKF_10', 'AOEI']#, 'TEnKF_3', 'TEnKF_5', 'TEnKF_10', 'AOEI']
x_col      = 'hx_dbz_local_mean_w'   # storm intensity proxy
y_col      = None   # RMSE reduction
n_bins     = 15
x_range    = (0, 20)
show_std   = True
savepath   = None

plot_binned_lines(
    df, methods,
    x_col=x_col, y_col=y_col,
    x_label='Storm intensity in zone — rho-weighted mean H(truth) [dBZ]',
    y_label='Mean RMSE change (analysis − prior) [dBZ]',
    n_bins=n_bins, x_range=x_range,
    show_std=show_std, hline=0,
    title='Storm intensity bins → mean RMSE change per method',
    savepath=savepath
)

In [ ]:
# =============================================================================
# PLOT 10 — 2D hexbin: prior departure × storm intensity → mean RMSE change
# Color = mean(rmse_a_obs_w - rmse_f_obs_w). Blue = improvement, red = degradation.
# mincnt=3 means bins with fewer than 3 points are hidden.
# =============================================================================

methods    = ['TEnKF_1', 'TEnKF_2', 'TEnKF_3','TEnKF_5','TEnKF_10', 'AOEI']#, 'TEnKF_3', 'TEnKF_5', 'TEnKF_10', 'AOEI']
x_range    = (-30, 60)   # dep_b
y_range    = (-1, 20)     # storm intensity
gridsize   = 30
vmin, vmax = -15, 15     # color range for RMSE change
one_figure = True
savepath   = None

plot_2d_hexbin_rmse(
    df, methods,
    x_range=x_range, y_range=y_range,
    gridsize=gridsize, vmin=vmin, vmax=vmax,
    one_figure=one_figure,
    title='Prior departure × storm intensity → mean RMSE change (blue = improvement)',
    savepath=savepath
)

In [ ]:
# =============================================================================
# PLOT 11 — same 2D hexbin colored by count
# Use a single method (all have the same x/y values) — just one panel.
# Shows where most of the data lives in the departure × storm intensity space.
# =============================================================================

methods    = ['TEnKF_1']   # one panel is enough — x/y axes are method-independent
x_range    = (-30, 60)
y_range    = (-1, 20)
gridsize   = 30
one_figure = True
savepath   = None

plot_2d_hexbin_count(
    df, methods,
    x_range=x_range, y_range=y_range,
    gridsize=gridsize, one_figure=one_figure,
    title='Point density: prior departure × storm intensity',
    savepath=savepath
)

In [ ]:
# =============================================================================
# PLOT 12 — marginal improvement per consecutive tempering step
# Each panel: rmse_a(Nt=n) - rmse_a(Nt=n+1)
# Red (positive) = adding one step helps. Blue (negative) = adding one step hurts.
# ntemp_sequence defines which consecutive pairs to compare.
# =============================================================================

ntemp_sequence = [1, 2,3,5,10]#, 3, 4,5, 6, 7, 8, 9, 10]   # pairs: 1→2, 2→3, 3→5, 5→10
x_range        = (-30, 60)
y_range        = (-1, 20)
gridsize       = 30
vmin, vmax     = -5, 5     # tighter range — marginal improvements are small
one_figure     = True
savepath       = None

plot_2d_hexbin_tempering_steps(
    df, ntemp_sequence,
    x_range=x_range, y_range=y_range,
    gridsize=gridsize, vmin=vmin, vmax=vmax,
    one_figure=one_figure,
    title='Marginal improvement per tempering step (red = adding step helps)',
    savepath=savepath
)

In [ ]:
# =============================================================================
# PLOT 13 — all Nt values vs Nt=1
# Color = mean( rmse_a(Nt=1) - rmse_a(Nt=n) ).
# Red = Nt=n is better than Nt=1. Blue = Nt=n is worse.
# vmin/vmax wider than Plot 12 to show cumulative differences.
# =============================================================================

ntemp_values   = [2, 3, 4, 5, 6, 7, 8, 9, 10]   # all vs Nt=1
x_range        = (-60, 60)
y_range        = (-1, 20)
gridsize       = 30
vmin, vmax     = -5, 5
one_figure     = True
savepath       = None

plot_2d_hexbin_vs_nt1(
    df, ntemp_values,
    x_range=x_range, y_range=y_range,
    gridsize=gridsize, vmin=vmin, vmax=vmax,
    one_figure=one_figure,
    title='RMSE improvement vs TEnKF Nt=1 (red = Nt=n better than Nt=1)',
    savepath=savepath
)

In [ ]:
# =============================================================================
# SUMMARY TABLE — mean ± std of key metrics per method
# =============================================================================

methods  = ['prior', 'TEnKF_1', 'TEnKF_2', 'TEnKF_3','TEnKF_5','TEnKF_10', 'AOEI']#, 'TEnKF_3', 'TEnKF_5', 'TEnKF_10', 'AOEI']
savepath = None

tbl = plot_summary_table(
    df, methods,
    title='Summary statistics per method (mean ± std)',
    savepath=savepath
)

tbl


In [ ]:
# =============================================================================
# PLOT 15 — Per-time obs-weighted RMSE statistics
# Requires mode='all_times' in Cell 2 (TIMES_LOAD controls which times appear).
# Top panel : absolute RMSE (prior --  vs analysis —) per method per time.
# Bottom panel: RMSE change (analysis − prior) per method per time.
# =============================================================================

SHOW_METHODS_TS = ['prior', 'AOEI', 'TEnKF_1', 'TEnKF_3', 'TEnKF_10']
TIMES_TS        = TIMES_ALL   # subset to e.g. ['1835','1840'] for a quick test

times_in_df = [t for t in TIMES_TS if t in df['time'].values]
x = list(range(len(times_in_df)))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Per-time obs-weighted RMSE statistics', fontsize=13)

for m_key in SHOW_METHODS_TS:
    rmse_f_vals, rmse_a_vals = [], []
    for t in times_in_df:
        sub = _get_method_df(df[df['time'] == t], m_key)
        rmse_f_vals.append(sub['rmse_f_obs_w'].mean() if len(sub) > 0 else np.nan)
        rmse_a_vals.append(sub['rmse_a_obs_w'].mean() if len(sub) > 0 else np.nan)
    c   = METHOD_COLORS.get(m_key, 'gray')
    lbl = METHOD_LABELS.get(m_key, m_key)
    axes[0].plot(x, rmse_f_vals, color=c, ls='--', marker='o', alpha=0.6, label=f'{lbl} prior')
    axes[0].plot(x, rmse_a_vals, color=c, ls='-',  marker='s', label=f'{lbl} analysis')
    if m_key != 'prior':
        drmse = [a - f for a, f in zip(rmse_a_vals, rmse_f_vals)]
        axes[1].plot(x, drmse, color=c, ls='-', marker='s', label=lbl)

axes[0].set_ylabel('Mean obs-weighted RMSE [dBZ]')
axes[0].legend(fontsize=7, ncol=3, loc='upper right')
axes[0].grid(True, linestyle=':', alpha=0.5)

axes[1].axhline(0, color='k', lw=1)
axes[1].set_ylabel('RMSE change (analysis − prior) [dBZ]')
axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, linestyle=':', alpha=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(times_in_df, rotation=45, ha='right')
axes[1].set_xlabel('Assimilation time (UTC)')

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# PLOT 16 — Per-time departure statistics
# Top panel   : mean O−B departure ± 1σ (same for all methods — obs distribution)
# Bottom panel: mean O−A departure per analysis method
# Together these show whether the obs are biased and if methods correct that bias.
# =============================================================================

SHOW_METHODS_DEP = ['AOEI', 'TEnKF_1', 'TEnKF_3', 'TEnKF_10']
TIMES_DEP        = TIMES_ALL   # subset to e.g. ['1835','1840'] for a quick test

times_in_df = [t for t in TIMES_DEP if t in df['time'].values]
x = list(range(len(times_in_df)))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Per-time departure statistics (O−B and O−A)', fontsize=13)

# Panel 1: mean O-B ± 1σ (prior only — same physical value for all methods)
ob_mean, ob_std = [], []
for t in times_in_df:
    sub = _get_method_df(df[df['time'] == t], 'prior')
    ob_mean.append(sub['dep_b'].mean() if len(sub) > 0 else np.nan)
    ob_std.append(sub['dep_b'].std()   if len(sub) > 0 else np.nan)
ob_mean = np.array(ob_mean)
ob_std  = np.array(ob_std)
axes[0].plot(x, ob_mean, 'k-o', lw=2, label='mean O−B')
axes[0].fill_between(x, ob_mean - ob_std, ob_mean + ob_std,
                     alpha=0.15, color='k', label='±1σ O−B')
axes[0].axhline(0, color='k', lw=0.5, ls=':')
axes[0].set_ylabel('O−B departure [dBZ]')
axes[0].legend(fontsize=8)
axes[0].grid(True, linestyle=':', alpha=0.5)

# Panel 2: mean O-A per analysis method
for m_key in SHOW_METHODS_DEP:
    oa_mean = []
    for t in times_in_df:
        sub = _get_method_df(df[df['time'] == t], m_key)
        oa_mean.append(sub['dep_a'].mean() if len(sub) > 0 else np.nan)
    c   = METHOD_COLORS.get(m_key, 'gray')
    lbl = METHOD_LABELS.get(m_key, m_key)
    axes[1].plot(x, oa_mean, color=c, ls='-', marker='s', lw=2, label=lbl)
axes[1].axhline(0, color='k', lw=1)
axes[1].set_ylabel('Mean O−A departure [dBZ]')
axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, linestyle=':', alpha=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(times_in_df, rotation=45, ha='right')
axes[1].set_xlabel('Assimilation time (UTC)')

plt.tight_layout()
plt.show()
